# 10c Classify Abstract Concepts

Classify each unique abstract once against the full reviewed catalog. The prompt contains only title, abstract, and concept definitions; it receives no SI values, pair split, or forecast outcomes. Results resume from a versioned parquet cache.

Jev receives only each abstract and the concept definitions. This run is configured to classify the full unique-abstract corpus selected in notebook 10a. Review its Jev request-count preflight before executing the model call cell.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipeline_config import cfg, cfg_path, load_pipeline_config
from amr.screening.system1 import OpenRouterJevClient
from amr.pipelines.concepts import build_si_concept_paths, load_concept_corpus, corpus_preflight
from amr.si.concepts import (
    SEED_CONCEPTS, concept_catalog_hash, estimate_classification_cost,
    classify_concept_abstracts_jev, build_yearly_concept_signals,
    rolling_origin_concept_forecasts, summarise_incremental_forecast_gains,
    propose_concept_children,
)

PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)
CONCEPT_PATHS = build_si_concept_paths(PROJECT_ROOT)
MODEL_NAME = str(cfg(PIPELINE_CONFIG, "system1.model", "typesafe/jev-1.13"))

corpus = pd.read_parquet(CONCEPT_PATHS["corpus"])
abstract_table = pd.read_parquet(CONCEPT_PATHS["abstracts"])
catalog = json.loads(CONCEPT_PATHS["catalog"].read_text())
CATALOG_VERSION = concept_catalog_hash(catalog)
MODEL_NAME = str(cfg(PIPELINE_CONFIG, "system1.model", "typesafe/jev-1.13"))
RUN_JEV_CLASSIFICATION = True
MAX_CALLS = None  # classify every pending unique abstract in the restricted corpus
MAX_WORKERS = int(cfg(PIPELINE_CONFIG, "system1.max_workers", 128))
abstracts = abstract_table.loc[abstract_table["abstract_available"], ["pmid", "title", "abstract"]].drop_duplicates("pmid")
print(f"Catalog {CATALOG_VERSION}; Jev model {MODEL_NAME}; {len(abstracts):,} unique abstracts; {len(catalog)} concepts")


## Preview prompt without making a request

Review the sample state and question template before enabling calls. The state contains title and abstract text only. Concept definitions and criteria are sent as typed questions; Jev receives no PMID, pathogen, antibiotic, SI value, split assignment, or future outcome.

In [ ]:
from amr.si.concepts import CONCEPT_SYSTEM_PROMPT
if len(abstracts):
    sample = abstracts.iloc[0]
    print(f"State sent to Jev:\nTitle: {sample['title']}\nAbstract: {sample['abstract'][:2000]}")
    print("Question template: Based only on the title and abstract, score explicit presence for the concept definition and criteria.")


## Optional resumable classification

Jev returns a NOUL score between 0 and 1 for each concept. Treat this as a model-assigned presence score, not a calibrated probability or a measure of importance. Blank abstracts are skipped. Successful scores are keyed by PMID, concept-definition hash, Jev model, and prompt version. When a catalog is expanded, unchanged concepts are reused; Jev is asked only for missing or revised concept labels, and cached values are linked into the active catalog version. Reruns skip labels already completed for the same concept.

`MAX_CALLS=None` processes all pending abstract/concept labels in a resumable run. `MAX_WORKERS` uses the configured System 1 worker count. `RUN_JEV_CLASSIFICATION=True` enables the Jev call cell; execution requires configured OpenRouter credentials. Progress checkpoints report elapsed time, ETA for this session, the latest OpenRouter balance, and an estimated ending balance once OpenRouter reports spend. If interrupted, rerun the cell to continue from cached scores.

In [ ]:
if RUN_JEV_CLASSIFICATION:
    jev = OpenRouterJevClient(model=MODEL_NAME)
    scores = classify_concept_abstracts_jev(
        abstracts, catalog, jev_client=jev, model_name=MODEL_NAME,
        cache_path=CONCEPT_PATHS["scores"], max_calls=MAX_CALLS, max_workers=MAX_WORKERS,
    )
    print(scores.groupby(["catalog_version", "status"]).size())
else:
    if CONCEPT_PATHS["scores"].exists():
        scores = pd.read_parquet(CONCEPT_PATHS["scores"])
        print(scores.groupby(["catalog_version", "status"]).size())
    else:
        print("Jev classification is disabled; no model calls were made.")
